# Homework 6 — Agentic Workflow

## Health Psychology & Chronic Back Pain Assistant

This notebook extends the existing Health Psychology RAG project with a controlled agentic workflow.

The assistant provides evidence-based psychoeducation about health psychology and chronic pain. In this demo, the agent is built for a healthcare provider such as [IVR](https://ivr.ua/), where neurologists, rehabilitation specialists, and other clinicians work with back pain and spine-related conditions.

For physical back-pain requests, the assistant does not diagnose or recommend treatment. Instead, it routes the user to professional care and adds a short biopsychosocial explanation.

The workflow follows:

`user goal → plan → action → observation → state update → final answer`.


## Stage 1 — Connect the repository and prepare the environment

In [3]:
from google.colab import userdata
from pathlib import Path
import os
import subprocess


GITHUB_USER = "swanksenia"
REPO_NAME = "health-psychology-rag-kb"

REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
PROJECT_ROOT = Path("/content") / REPO_NAME


github_token = userdata.get("GITHUB_TOKEN")

if not github_token:
    raise ValueError(
        "GITHUB_TOKEN was not found in Colab Secrets."
    )


# Temporary helper for secure GitHub authentication.
askpass_path = Path("/content/git_askpass.sh")

askpass_path.write_text(
    """#!/bin/sh
case "$1" in
    *Username*) echo "x-access-token" ;;
    *Password*) echo "$GITHUB_TOKEN" ;;
esac
""",
    encoding="utf-8",
)

askpass_path.chmod(0o700)


git_environment = os.environ.copy()
git_environment["GITHUB_TOKEN"] = github_token
git_environment["GIT_ASKPASS"] = str(askpass_path)
git_environment["GIT_TERMINAL_PROMPT"] = "0"


if (PROJECT_ROOT / ".git").exists():
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "pull"],
        check=True,
        env=git_environment,
    )
else:
    subprocess.run(
        ["git", "clone", REPO_URL, str(PROJECT_ROOT)],
        check=True,
        env=git_environment,
    )


print("Project root:", PROJECT_ROOT)

Project root: /content/health-psychology-rag-kb


In [4]:
subprocess.run(
    [
        "git",
        "-C",
        str(PROJECT_ROOT),
        "config",
        "user.name",
        "Kseniia Lebedieva",
    ],
    check=True,
)

subprocess.run(
    [
        "git",
        "-C",
        str(PROJECT_ROOT),
        "config",
        "user.email",
        "lebedevaky@gmail.com",
    ],
    check=True,
)

print("Git identity configured.")

Git identity configured.


## Stage 2 — Define agent state and business rules

This stage defines the shared workflow state and the core business rules for the assistant.

The state stores the user goal, selected route, execution plan, tool calls, observations, intermediate results, and final answer.

The assistant follows two core rules:

- health psychology questions are answered using the scientific knowledge base;
- physical back-pain or spine-related treatment requests are not diagnosed or treated by the assistant and are routed to professional IVR care.

In [41]:
from typing import Any, Dict


IVR_CARE = {
    "provider": "Institute of Vertebrology and Rehabilitation",
    "website": "https://ivr.ua/",
    "country": "Ukraine",
    "cities": [
        "Kyiv",
        "Lviv",
        "Ivano-Frankivsk",
    ],
    "care_categories": [
        "doctor consultations",
        "therapeutic massage",
        "physical therapy and physiotherapy",
        "rehabilitation",
        "manual and other medical procedures",
        "injection therapy",
        "diagnostics",
        "orthopedic corrective devices",
    ],
    "online_consultation": True,
}


def create_initial_state(user_message: str) -> Dict[str, Any]:
    return {
        "user_message": user_message,
        "user_goal": None,

        "selected_route": None,

        "plan": [],
        "current_step": None,
        "completed_steps": [],

        "health_psychology_topics": [],
        "physical_back_pain_request": False,
        "spine_condition_detected": False,
        "medical_advice_requested": False,

        "care_pathway_needed": False,
        "care_mode": None,

        "tool_calls": [],
        "observations": [],
        "intermediate_results": {},

        "fallback_used": False,
        "final_answer": None,
    }

In [53]:
state = create_initial_state(
    "My back hurts, and when I go to the gym to strengthen my back, "
    "the pain gets even worse. This did not happen before."
)

state

{'user_message': 'My back hurts, and when I go to the gym to strengthen my back, the pain gets even worse. This did not happen before.',
 'user_goal': None,
 'selected_route': None,
 'plan': [],
 'current_step': None,
 'completed_steps': [],
 'health_psychology_topics': [],
 'physical_back_pain_request': False,
 'spine_condition_detected': False,
 'medical_advice_requested': False,
 'care_pathway_needed': False,
 'care_mode': None,
 'tool_calls': [],
 'observations': [],
 'intermediate_results': {},
 'fallback_used': False,
 'final_answer': None}

## Stage 3 — Define agent tools

This stage defines two read-only tools used by the workflow.

- `search_knowledge_base()` reuses the existing retrieval pipeline from Homework 3 to search the scientific Health Psychology knowledge base and return relevant evidence with source metadata.
- `get_ivr_care_options()` returns structured IVR care options for users with physical back-pain or spine-related requests.

Both tools return structured results that will later be stored as observations in the workflow state.

In [10]:
# Install retrieval dependencies used in Homework 3.
!pip install -q --no-cache-dir faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 318.9 MB/s eta 0:00:00


In [11]:
import sys
import faiss

from sentence_transformers import SentenceTransformer

sys.path.append(str(PROJECT_ROOT))

from scripts.retrieval_improved import (
    CHUNKS_PATH,
    INDEX_PATH,
    MODEL_NAME,
    TOP_K,
    load_jsonl,
    add_metadata,
    semantic_search,
)

In [12]:
# Load the existing Health Psychology knowledge base and FAISS index.

retrieval_chunks = load_jsonl(CHUNKS_PATH)
add_metadata(retrieval_chunks)

retrieval_index = faiss.read_index(str(INDEX_PATH))

if retrieval_index.ntotal != len(retrieval_chunks):
    raise ValueError(
        "FAISS index size does not match the number of retrieval chunks."
    )

retrieval_model = SentenceTransformer(MODEL_NAME)

print("Retrieval tool ready.")
print("Chunks:", len(retrieval_chunks))
print("FAISS vectors:", retrieval_index.ntotal)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Retrieval tool ready.
Chunks: 474
FAISS vectors: 474


In [54]:
def search_knowledge_base(
    query: str,
    top_k: int = TOP_K,
) -> Dict[str, Any]:
    results = semantic_search(
        query=query,
        model=retrieval_model,
        index=retrieval_index,
        chunks=retrieval_chunks,
        top_k=top_k,
    )

    return {
        "success": bool(results),
        "query": query,
        "results": results,
    }

In [55]:
def get_ivr_care_options() -> Dict[str, Any]:
    return {
        "success": True,
        "provider": IVR_CARE["provider"],
        "website": IVR_CARE["website"],
        "country": IVR_CARE["country"],
        "cities": IVR_CARE["cities"],
        "care_categories": IVR_CARE["care_categories"],
        "online_consultation": IVR_CARE["online_consultation"],
    }

In [56]:
knowledge_result = search_knowledge_base(
    "What is the biopsychosocial model of health?"
)

ivr_result = get_ivr_care_options()

print("Knowledge base result:")
print(knowledge_result)

print("\nIVR care result:")
print(ivr_result)

Knowledge base result:
{'success': True, 'query': 'What is the biopsychosocial model of health?', 'results': [{'score': 0.7959496974945068, 'chunk_id': 'ogden_2019_health_psychology__0013', 'text': 'smoking), pressures to change behavior (e.g. peer group expectations, parental pressure), social values on health (e.g. whether health was regarded as a good or a bad thing), social class, the environment, and ethnicity. #### Fig 1 The biopsychosocial model of health and illness (after Engel 1977, 1980) ![Fig 1 The biopsychosocial model of health and illness (after Engel 1977, 1980)](assets/ogden_p7_figure_1.png) Figure description: The biopsychosocial model explains health and illness through the interaction of biological, psychological, and social factors.', 'metadata': {'document_id': 'ogden_2019_health_psychology', 'source_file': 'data/raw/ogden_2019_health_psychology.pdf', 'section': '1.The Biopsychosocial Model', 'chunk_index': 13, 'document_type': 'textbook'}}, {'score': 0.7779133319

## Stage 4 — Classify the request and build the plan

This stage classifies the user request using deterministic rules and builds the execution plan for the workflow.

The router distinguishes between:

- health psychology questions;
- physical back-pain or spine-related requests;
- unclear requests that require clarification.

The selected route and execution plan are stored in the shared state.

In [72]:
def classify_request(state: Dict[str, Any]) -> Dict[str, Any]:
    message = state["user_message"].lower()

    back_pain_keywords = [
        "lower back pain",
        "back pain",
        "sciatica pain",
        "upper back ache",
        "sharp pain in spine",
        "stiff lower back",
        "neck pain",
        "herniated disc",
        "bulging disc",
        "pinched nerve",
        "muscle strain",
        "spinal stenosis",
        "degenerative disc disease",
        "disc hernia",
        "spinal hernia",
        "lumbar pain",
        "lumbar spine",
        "pain down the leg",
        "shooting pain",
        "radiating pain",
        "back stiffness",
        "neck stiffness",
        "back hurts",
        "pain gets worse",
    ]

    medical_advice_keywords = [
        "back pain relief",
        "back pain treatment",
        "physical therapy for back",
        "chiropractor near me",
        "back stretches for pain",
        "pain relief medicine",
        "painkiller",
        "what should i do",
        "what can i do",
        "how should i treat",
        "how to treat",
        "what treatment",
        "what exercises",
        "which exercises",
        "what medication",
        "what medicine",
    ]

    psychology_keywords = [
        "health psychology",
        "biopsychosocial",
        "stress",
        "anxiety",
        "depression",
        "mood",
        "mental health",
        "coping",
        "fear",
        "pain",
        "can't work",
        "cannot work",
        "can't concentrate",
        "cannot concentrate",
        "angry",
        "can't sleep",
        "cannot sleep",
        "sleep problems",
        "isolated",
        "avoid people",
        "don't want to see anyone",
        "behavior",
        "harmful behavior",
        "harmful behaviour",
        "behavior change",
        "behaviour change",
    ]

    spine_condition_keywords = [
        "spine",
        "herniated disc",
        "bulging disc",
        "disc hernia",
        "spinal hernia",
        "sciatica",
        "pinched nerve",
        "spinal stenosis",
        "degenerative disc disease",
        "lumbar spine",
    ]

    state["current_step"] = "classify_request"

    state["physical_back_pain_request"] = any(
        keyword in message
        for keyword in back_pain_keywords
    )

    state["spine_condition_detected"] = any(
        keyword in message
        for keyword in spine_condition_keywords
    )

    state["medical_advice_requested"] = any(
        keyword in message
        for keyword in medical_advice_keywords
    )

    detected_topics = [
        keyword
        for keyword in psychology_keywords
        if keyword in message
    ]

    state["health_psychology_topics"] = detected_topics

    if state["physical_back_pain_request"]:
        state["selected_route"] = "back_pain_medical_request"
        state["user_goal"] = (
            "understand a physical back-pain problem and what to do next"
        )

    elif detected_topics:
        state["selected_route"] = "psychoeducation"
        state["user_goal"] = (
            "understand a health psychology topic using scientific evidence"
        )

    else:
        state["selected_route"] = "clarification"
        state["user_goal"] = "clarify the user's request"

    state["completed_steps"].append("classify_request")

    return state


def build_plan(state: Dict[str, Any]) -> Dict[str, Any]:
    state["current_step"] = "build_plan"

    if state["selected_route"] == "back_pain_medical_request":
        state["plan"] = [
            "search_knowledge_base",
            "apply_medical_boundary",
            "get_ivr_care_options",
            "add_biopsychosocial_context",
            "build_final_answer",
        ]

        state["care_pathway_needed"] = True

    elif state["selected_route"] == "psychoeducation":
        state["plan"] = [
            "search_knowledge_base",
            "build_final_answer",
        ]

    else:
        state["plan"] = [
            "build_clarification_answer",
        ]

    state["completed_steps"].append("build_plan")

    return state

In [73]:
state = create_initial_state(
    "My back hurts, and when I go to the gym to strengthen my back, "
    "the pain gets even worse. This did not happen before. What should I do?"
)

state = classify_request(state)
state = build_plan(state)

state

{'user_message': 'My back hurts, and when I go to the gym to strengthen my back, the pain gets even worse. This did not happen before. What should I do?',
 'user_goal': 'understand a physical back-pain problem and what to do next',
 'selected_route': 'back_pain_medical_request',
 'plan': ['search_knowledge_base',
  'apply_medical_boundary',
  'get_ivr_care_options',
  'add_biopsychosocial_context',
  'build_final_answer'],
 'current_step': 'build_plan',
 'completed_steps': ['classify_request', 'build_plan'],
 'health_psychology_topics': ['pain'],
 'physical_back_pain_request': True,
 'spine_condition_detected': False,
 'medical_advice_requested': True,
 'care_pathway_needed': True,
 'care_mode': None,
 'tool_calls': [],
 'observations': [],
 'intermediate_results': {},
 'fallback_used': False,
 'final_answer': None}

## Stage 5 — Execute workflow steps and record observations

This stage executes the planned workflow steps and stores their results in the shared state.

Each tool call is recorded in `tool_calls`, its result is stored as an `observation`, and important outputs are saved in `intermediate_results` for use by later steps.

For back-pain requests, the workflow:

1. retrieves scientific evidence from the Health Psychology knowledge base;
2. applies the medical boundary;
3. retrieves IVR care options;
4. prepares a short biopsychosocial context grounded in the retrieved evidence.

In [74]:
def run_knowledge_search(state: Dict[str, Any]) -> Dict[str, Any]:
    state["current_step"] = "search_knowledge_base"

    retrieval_query = (
        f"{state['user_message']} "
        "biopsychosocial model chronic pain health psychology"
    )

    state["tool_calls"].append({
        "tool": "search_knowledge_base",
        "input": {
            "query": retrieval_query,
        },
    })

    result = search_knowledge_base(retrieval_query)

    observation = {
        "tool": "search_knowledge_base",
        "success": result["success"],
        "result_count": len(result.get("results", [])),
        "results": result.get("results", []),
    }

    state["observations"].append(observation)
    state["intermediate_results"]["knowledge_search"] = result

    if not result["success"]:
        state["fallback_used"] = True

    state["completed_steps"].append("search_knowledge_base")

    return state

In [75]:
def apply_medical_boundary(state: Dict[str, Any]) -> Dict[str, Any]:
    state["current_step"] = "apply_medical_boundary"

    boundary = {
        "diagnosis_allowed": False,
        "treatment_recommendations_allowed": False,
        "medical_referral_required": (
            state["physical_back_pain_request"]
            or state["spine_condition_detected"]
        ),
        "psychoeducation_allowed": True,
    }

    state["intermediate_results"]["medical_boundary"] = boundary

    state["observations"].append({
        "step": "apply_medical_boundary",
        "result": boundary,
    })

    state["completed_steps"].append("apply_medical_boundary")

    return state

In [76]:
def run_ivr_care_tool(state: Dict[str, Any]) -> Dict[str, Any]:
    state["current_step"] = "get_ivr_care_options"

    state["tool_calls"].append({
        "tool": "get_ivr_care_options",
        "input": {},
    })

    result = get_ivr_care_options()

    observation = {
        "tool": "get_ivr_care_options",
        "success": result["success"],
        "provider": result["provider"],
        "cities": result["cities"],
        "online_consultation": result["online_consultation"],
    }

    state["observations"].append(observation)
    state["intermediate_results"]["ivr_care"] = result

    state["completed_steps"].append("get_ivr_care_options")

    return state

In [77]:
def add_biopsychosocial_context(
    state: Dict[str, Any],
) -> Dict[str, Any]:
    state["current_step"] = "add_biopsychosocial_context"

    knowledge_result = state["intermediate_results"].get(
        "knowledge_search",
        {},
    )

    results = knowledge_result.get("results", [])

    if not results:
        state["intermediate_results"]["biopsychosocial_context"] = None
        state["fallback_used"] = True

    else:
        top_result = results[0]

        context = {
            "text": top_result["text"],
            "score": top_result["score"],
            "chunk_id": top_result["chunk_id"],
            "source": top_result["metadata"],
        }

        state["intermediate_results"][
            "biopsychosocial_context"
        ] = context

    state["completed_steps"].append(
        "add_biopsychosocial_context"
    )

    return state

In [78]:
state = run_knowledge_search(state)
state = apply_medical_boundary(state)
state = run_ivr_care_tool(state)
state = add_biopsychosocial_context(state)

state

{'user_message': 'My back hurts, and when I go to the gym to strengthen my back, the pain gets even worse. This did not happen before. What should I do?',
 'user_goal': 'understand a physical back-pain problem and what to do next',
 'selected_route': 'back_pain_medical_request',
 'plan': ['search_knowledge_base',
  'apply_medical_boundary',
  'get_ivr_care_options',
  'add_biopsychosocial_context',
  'build_final_answer'],
 'current_step': 'add_biopsychosocial_context',
 'completed_steps': ['classify_request',
  'build_plan',
  'search_knowledge_base',
  'apply_medical_boundary',
  'get_ivr_care_options',
  'add_biopsychosocial_context'],
 'health_psychology_topics': ['pain'],
 'physical_back_pain_request': True,
 'spine_condition_detected': False,
 'medical_advice_requested': True,
 'care_pathway_needed': True,
 'care_mode': None,
 'tool_calls': [{'tool': 'search_knowledge_base',
   'input': {'query': 'My back hurts, and when I go to the gym to strengthen my back, the pain gets even w

## Stage 6 — Build the final answer

This stage generates the user-facing response from the accumulated workflow state.

The final answer follows the selected route:

- psychoeducation requests are answered using retrieved scientific evidence;
- physical back-pain requests receive a medical boundary, IVR care information, and a short evidence-based explanation of pain from a health psychology perspective;
- unclear requests receive a clarification question.

In [79]:
def build_final_answer(state: Dict[str, Any]) -> Dict[str, Any]:
    state["current_step"] = "build_final_answer"

    route = state["selected_route"]

    if route == "back_pain_medical_request":
        knowledge_result = state["intermediate_results"].get(
            "knowledge_search",
            {},
        )

        ivr_care = state["intermediate_results"].get(
            "ivr_care",
            {},
        )

        pain_context = state["intermediate_results"].get(
            "biopsychosocial_context",
        )

        answer_parts = [
            (
                "I cannot diagnose the cause of your back pain or recommend "
                "treatment based on this message. Since the pain is new or "
                "becoming worse, it would be appropriate to discuss it with "
                "a healthcare professional."
            )
        ]

        if ivr_care:
            cities = ", ".join(ivr_care["cities"])

            answer_parts.append(
                (
                    f"You can consult specialists at the"
                    f"{ivr_care['provider']}. IVR provides care in "
                    f"{cities}, and online consultations are also available. "
                    f"{ivr_care['website']}"
                )
            )

        if pain_context:
            answer_parts.append(
                (
                  "From a health psychology perspective, pain is influenced not only "
                  "by biological factors but also by psychological and behavioral "
                  "processes. The retrieved evidence describes how fear of pain and "
                  "avoidance of painful activities can become part of a cycle that "
                  "affects how pain is experienced over time."
                )
            )

            source = pain_context["source"]

            answer_parts.append(
                (
                    f"Source: {source['document_id']} "
                    f"({source['document_type']}, "
                    f"section: {source['section']})."
                )
            )

        state["final_answer"] = "\n\n".join(answer_parts)

    elif route == "psychoeducation":
        knowledge_result = state["intermediate_results"].get(
            "knowledge_search",
            {},
        )

        results = knowledge_result.get("results", [])

        if results:
            top_result = results[0]

            state["final_answer"] = (
                f"{top_result['text']}\n\n"
                f"Source: {top_result['metadata']['document_id']} "
                f"({top_result['metadata']['document_type']}, "
                f"section: {top_result['metadata']['section']})."
            )

        else:
            state["fallback_used"] = True
            state["final_answer"] = (
                "I could not find enough relevant evidence in the current "
                "knowledge base to answer this question reliably."
            )

    else:
        state["final_answer"] = (
            "Could you clarify what you would like to understand? "
            "You can ask about health psychology, chronic pain, emotional "
            "responses to illness, or a back-pain concern."
        )

    state["completed_steps"].append("build_final_answer")

    return state

In [80]:
state = build_final_answer(state)

print(state["final_answer"])

I cannot diagnose the cause of your back pain or recommend treatment based on this message. Since the pain is new or becoming worse, it would be appropriate to discuss it with a healthcare professional.

You can consult specialists at theInstitute of Vertebrology and Rehabilitation. IVR provides care in Kyiv, Lviv, Ivano-Frankivsk, and online consultations are also available. https://ivr.ua/

From a health psychology perspective, pain is influenced not only by biological factors but also by psychological and behavioral processes. The retrieved evidence describes how fear of pain and avoidance of painful activities can become part of a cycle that affects how pain is experienced over time.

Source: wright_2019_3p_disease_model (research_article, section: Examples of 3P-Health Model Applications to Specific Diseases).


## Stage 7 — Build the complete workflow runner

This stage combines all previous steps into one controlled agent workflow.

The runner:

1. creates the initial state;
2. classifies the request;
3. builds the execution plan;
4. executes the planned steps;
5. updates the shared state after each step;
6. returns the final state with the final answer.

In [117]:
def run_agent(user_message: str) -> Dict[str, Any]:
    state = create_initial_state(user_message)

    state = classify_request(state)
    state = build_plan(state)

    for step in state["plan"]:
        if step == "search_knowledge_base":
            state = run_knowledge_search(state)

        elif step == "apply_medical_boundary":
            state = apply_medical_boundary(state)

        elif step == "get_ivr_care_options":
            state = run_ivr_care_tool(state)

        elif step == "add_biopsychosocial_context":
            state = add_biopsychosocial_context(state)

        elif step == "build_final_answer":
            state = build_final_answer(state)

        elif step == "build_clarification_answer":
            state = build_clarification_answer(state)

        else:
            state["fallback_used"] = True
            state["observations"].append({
                "step": step,
                "success": False,
                "error": f"Unknown workflow step: {step}",
            })

    return state

In [118]:
final_state = run_agent(
    "My back hurts, and when I go to the gym to strengthen my back, "
    "the pain gets even worse. This did not happen before. What should I do?"
)

print(final_state["final_answer"])

I cannot diagnose the cause of your back pain or recommend medication or treatment based on this message. It would be appropriate to discuss your symptoms with a healthcare professional.

You can consult specialists at the Institute of Vertebrology and Rehabilitation. IVR provides care in Kyiv, Lviv, Ivano-Frankivsk, and online consultations are also available. https://ivr.ua/

Pain perception is influenced by an interplay of cognitive, emotional, learning, and behavioral processes, including anxiety. Individual responses to pain—such as altering posture, expressing distress, or avoiding activity—can be positively reinforced through attention, sympathy, or secondary gains like time off work. Consequently, this reinforcement and the resulting inactivity or adoption of a sick role can increase the perception of pain.


In [119]:
print("Route:")
print(final_state["selected_route"])

print("\nPlan:")
print(final_state["plan"])

print("\nCompleted steps:")
print(final_state["completed_steps"])

print("\nTool calls:")
print(final_state["tool_calls"])

print("\nFinal answer:")
print(final_state["final_answer"])

Route:
back_pain_medical_request

Plan:
['search_knowledge_base', 'apply_medical_boundary', 'get_ivr_care_options', 'add_biopsychosocial_context', 'build_final_answer']

Completed steps:
['classify_request', 'build_plan', 'search_knowledge_base', 'apply_medical_boundary', 'get_ivr_care_options', 'add_biopsychosocial_context', 'build_final_answer']

Tool calls:
[{'tool': 'search_knowledge_base', 'input': {'query': 'My back hurts, and when I go to the gym to strengthen my back, the pain gets even worse. This did not happen before. What should I do? psychological behavioral aspects of pain pain perception health psychology'}}, {'tool': 'get_ivr_care_options', 'input': {}}]

Final answer:
I cannot diagnose the cause of your back pain or recommend medication or treatment based on this message. It would be appropriate to discuss your symptoms with a healthcare professional.

You can consult specialists at the Institute of Vertebrology and Rehabilitation. IVR provides care in Kyiv, Lviv, Ivan

## Stage 8 — Test the workflow on 5 scenarios

This stage tests the complete agent workflow on five representative scenarios.

The examples cover:

- health psychology psychoeducation;
- chronic pain and functional impact;
- physical back-pain requests;
- medication-related medical advice requests;
- unclear psychological complaints.

For each case, the workflow records the selected route, plan, tool calls, observations, completed steps, and final answer.

In [120]:
test_cases = [
    "How to change harmful behavior?",

    "Why can't I work effectively with chronic pain?",

    (
        "My back hurts, and when I go to the gym to strengthen my back, "
        "the pain gets even worse. What should I do?"
    ),

    (
        "I have terrible lower back pain. "
        "Which painkillers should I use?"
    ),

    "I feel tired and annoyed lately and I don't know why.",
]

In [129]:
test_results = []

for i, question in enumerate(test_cases, start=1):
    result = run_agent(question)

    test_results.append({
        "case": i,
        "question": question,
        "state": result,
    })

    print("=" * 100)
    print(f"CASE {i}")
    print("=" * 100)

    print("\nQuestion:")
    print(question)

    print("\nRoute:")
    print(result["selected_route"])

    print("\nPlan:")
    print(result["plan"])

    print("\nCompleted steps:")
    print(result["completed_steps"])

    print("\nTool calls:")
    print(result["tool_calls"])

    print("\nObservations:")
    print(result["observations"])

    print("\nFinal answer:")
    print(result["final_answer"])

    print()

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 27.107713336s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.6-flash'},

**Test result:**
The workflow and routing work as expected, but the retrieval logic needs improvement.

For general psychoeducation questions, the agent currently adds chronic-pain context to every search query. This can bias retrieval toward pain-related sources, even when the user asks about another health psychology topic.

**Next step:**

* use the original user question for general psychoeducation;
* add pain-specific context only for back-pain or medical requests;
* route unclear requests to clarification instead of searching the knowledge base immediately.


### Stage 8.1 — Improve routing and retrieval after testing

The initial tests showed that the workflow structure works correctly, but the retrieval query was biased toward chronic-pain content.

This stage improves the retrieval logic:

- general psychoeducation requests search the knowledge base using the original user question;
- back-pain medical requests use additional pain-psychology context;
- unclear requests are routed to clarification without retrieval.

In [86]:
def build_retrieval_query(state: Dict[str, Any]) -> str:
    user_message = state["user_message"]

    if state["selected_route"] == "back_pain_medical_request":
        return (
            f"{user_message} "
            "psychological behavioral aspects of pain "
            "pain perception health psychology"
        )

    return user_message


def run_knowledge_search(state: Dict[str, Any]) -> Dict[str, Any]:
    state["current_step"] = "search_knowledge_base"

    retrieval_query = build_retrieval_query(state)

    state["tool_calls"].append({
        "tool": "search_knowledge_base",
        "input": {
            "query": retrieval_query,
        },
    })

    result = search_knowledge_base(retrieval_query)

    observation = {
        "tool": "search_knowledge_base",
        "success": result["success"],
        "result_count": len(result.get("results", [])),
        "results": result.get("results", []),
    }

    state["observations"].append(observation)
    state["intermediate_results"]["knowledge_search"] = result

    if not result["success"]:
        state["fallback_used"] = True

    state["completed_steps"].append("search_knowledge_base")

    return state

In [87]:
test_results = []

for i, question in enumerate(test_cases, start=1):
    result = run_agent(question)

    test_results.append({
        "case": i,
        "question": question,
        "state": result,
    })

    print("=" * 100)
    print(f"CASE {i}")
    print("=" * 100)

    print("\nQuestion:")
    print(question)

    print("\nRoute:")
    print(result["selected_route"])

    print("\nPlan:")
    print(result["plan"])

    print("\nCompleted steps:")
    print(result["completed_steps"])

    print("\nTool calls:")
    print(result["tool_calls"])

    print("\nObservations:")
    print(result["observations"])

    print("\nFinal answer:")
    print(result["final_answer"])

    print()

CASE 1

Question:
How to change harmful behavior?

Route:
psychoeducation

Plan:
['search_knowledge_base', 'build_final_answer']

Completed steps:
['classify_request', 'build_plan', 'search_knowledge_base', 'build_final_answer']

Tool calls:
[{'tool': 'search_knowledge_base', 'input': {'query': 'How to change harmful behavior?'}}]

Observations:
[{'tool': 'search_knowledge_base', 'success': True, 'result_count': 3, 'results': [{'score': 0.6106940507888794, 'chunk_id': 'michie_2011_behaviour_change_wheel__0021', 'text': 'is too complex and that the constructs too ill-defined to be able to establish a useful, scientifically-based framework. Another is that no framework can address the level of detail required to determine what will or will not be an effective intervention. The response to this is twofold: these are empirical questions and there is already evidence that characterising interventions by behaviour change techniques (BCTs) can be helpful in understanding which interventions a

### Stage 8.2 — Add LLM synthesis for the final answer

In [89]:
!pip install -q -U google-genai

In [99]:
from google.genai import types

In [130]:
from google.genai.errors import ClientError

In [100]:
from google import genai
from google.colab import userdata

gemini_api_key = userdata.get("GEMINI_API_KEY")

if not gemini_api_key:
    raise ValueError(
        "GEMINI_API_KEY was not found in Colab Secrets."
    )

gemini_client = genai.Client(
    api_key=gemini_api_key
)

print("Gemini client ready.")

Gemini client ready.


In [122]:
def synthesize_with_gemini(
    state: Dict[str, Any],
) -> str:

    knowledge_result = state[
        "intermediate_results"
    ].get("knowledge_search", {})

    results = knowledge_result.get("results", [])

    evidence_text = "\n\n".join(
        [
            (
                f"Source {i + 1}: "
                f"{item['metadata']['document_id']}\n"
                f"{item['text']}"
            )
            for i, item in enumerate(results)
        ]
    )

    prompt = f"""
You are an evidence-based Health Psychology assistant.

User question:
{state["user_message"]}

Selected route:
{state["selected_route"]}

Retrieved scientific evidence:
{evidence_text}

Instructions:
- Answer the user's question briefly and clearly.
- Use only information supported by the retrieved evidence.
- Do not invent facts.
- Do not diagnose.
- Do not recommend medications or physical treatment.
- If the request is about physical back pain or medical treatment,
  state that medical advice should come from a healthcare professional.
- For back-pain medical requests, include a very short explanation
  of the psychological or behavioral aspects of pain supported by
  the retrieved evidence.
- Keep the answer concise and user-friendly.
"""

    response = gemini_client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt,
    )

    return response.text

In [123]:
def build_final_answer(state: Dict[str, Any]) -> Dict[str, Any]:
    state["current_step"] = "build_final_answer"

    route = state["selected_route"]

    if route == "back_pain_medical_request":
        ivr_care = state["intermediate_results"].get(
            "ivr_care",
            {},
        )

        llm_answer = synthesize_with_gemini(state)

        answer_parts = [
            (
                "I cannot diagnose the cause of your back pain or recommend "
                "medication or treatment based on this message. "
                "It would be appropriate to discuss your symptoms with "
                "a healthcare professional."
            )
        ]

        if ivr_care:
            cities = ", ".join(ivr_care["cities"])

            answer_parts.append(
                (
                    f"You can consult specialists at the "
                    f"{ivr_care['provider']}. "
                    f"IVR provides care in {cities}, "
                    f"and online consultations are also available. "
                    f"{ivr_care['website']}"
                )
            )

        if llm_answer:
            answer_parts.append(llm_answer)

        state["final_answer"] = "\n\n".join(answer_parts)

    elif route == "psychoeducation":
        state["final_answer"] = synthesize_with_gemini(state)

    state["completed_steps"].append(
        "build_final_answer"
    )

    return state

In [124]:
def build_clarification_answer(
    state: Dict[str, Any],
) -> Dict[str, Any]:
    state["current_step"] = "build_clarification_answer"

    state["final_answer"] = (
        "Could you clarify what you would like to understand? "
        "You can ask about health psychology, chronic pain, "
        "emotional responses to illness, or a back-pain concern."
    )

    state["completed_steps"].append(
        "build_clarification_answer"
    )

    return state

In [134]:
def synthesize_with_gemini(
    state: Dict[str, Any],
) -> str:

    knowledge_result = state[
        "intermediate_results"
    ].get("knowledge_search", {})

    results = knowledge_result.get("results", [])

    evidence_text = "\n\n".join(
        [
            (
                f"Source {i + 1}\n"
                f"Chunk ID: {item['chunk_id']}\n"
                f"Document: {item['metadata']['document_id']}\n"
                f"Evidence: {item['text']}"
            )
            for i, item in enumerate(results)
        ]
    )

    state["intermediate_results"]["llm_evidence"] = evidence_text

    if state["selected_route"] == "back_pain_medical_request":
        task = """
    Rewrite only the parts of the retrieved evidence that explain
    psychological, emotional, cognitive, learning, or behavioral aspects
    of pain and pain perception.

    Do not include any information about treatment, therapy, medication,
    exercise, pain-management techniques, interventions, biofeedback,
    relaxation, hypnosis, or other methods of reducing pain.

    The medical/treatment part of the user's request is handled separately
    by the workflow through professional referral.

    Use 2-3 short sentences.
    Paraphrase only what is explicitly present in the retrieved evidence.
    Do not add interpretations, recommendations, or new information.
    """
    else:
        task = """
    Use the retrieved evidence to answer the user's question as directly
    as the evidence allows.

    Summarize and combine relevant statements from the evidence into
    a concise, readable answer.
    """

    prompt = f"""
You are the synthesis layer of a Health Psychology RAG system.

User question:
{state["user_message"]}

Retrieved evidence:
{evidence_text}

Task:
{task}

Rules:
- Use only information contained in the retrieved evidence.
- Do not use general knowledge.
- Do not add new facts, explanations, examples, recommendations,
  causal claims, or interpretations.
- Preserve the meaning of the evidence.
- You may paraphrase, shorten, and combine evidence from multiple chunks.
- Remove broken links, citation artifacts, incomplete sentence fragments,
  and irrelevant text.
- Keep the answer concise and easy to read.
- Do not mention "the provided evidence" or "the knowledge base"
  unless there is genuinely no relevant information in any retrieved chunk.
"""

    try:
      response = gemini_client.models.generate_content(
          model="gemini-3.6-flash",
          contents=prompt,
          config=types.GenerateContentConfig(
              automatic_function_calling=
                  types.AutomaticFunctionCallingConfig(
                      disable=True
                  )
          ),
      )

      return response.text

    except ClientError as error:
          if error.code == 429:
              state["fallback_used"] = True

              return (
                  "LLM synthesis is temporarily unavailable because the API "
                  "quota was exceeded. Relevant evidence was successfully "
                  "retrieved from the knowledge base."
              )

          raise

In [135]:
test_state = run_agent(
    "How to change harmful behavior?"
)

print("Evidence sent to Gemini:")
print(test_state["intermediate_results"]["llm_evidence"])

print("\nGemini final answer:")
print(test_state["final_answer"])

Evidence sent to Gemini:
Source 1
Chunk ID: michie_2011_behaviour_change_wheel__0021
Document: michie_2011_behaviour_change_wheel
Evidence: is too complex and that the constructs too ill-defined to be able to establish a useful, scientifically-based framework. Another is that no framework can address the level of detail required to determine what will or will not be an effective intervention. The response to this is twofold: these are empirical questions and there is already evidence that characterising interventions by behaviour change techniques (BCTs) can be helpful in understanding which interventions are more or less effective [6](https://pmc.ncbi.nlm.nih.gov/articles/PMC3096582/#B6),[17](https://pmc.ncbi.nlm.nih.gov/articles/PMC3096582/#B17); and not to embark on this enterprise is to give up on achieving a science of behaviour change before the first hurdle and condemn this field to opinion and fashion.

Source 2
Chunk ID: ogden_2019_health_psychology__0068
Document: ogden_2019_

## Stage 8.3 — Re-run the 5 scenarios with LLM synthesis

In [136]:
test_results = []

for i, question in enumerate(test_cases, start=1):
    result = run_agent(question)

    test_results.append({
        "case": i,
        "question": question,
        "state": result,
    })

    print("=" * 100)
    print(f"CASE {i}")
    print("=" * 100)

    print("\nQuestion:")
    print(question)

    print("\nRoute:")
    print(result["selected_route"])

    print("\nPlan:")
    print(result["plan"])

    print("\nCompleted steps:")
    print(result["completed_steps"])

    print("\nTool calls:")
    print(result["tool_calls"])

    print("\nRetrieved chunk IDs:")
    knowledge_search = result["intermediate_results"].get(
        "knowledge_search",
        {},
    )

    for item in knowledge_search.get("results", []):
        print("-", item["chunk_id"])

    print("\nFinal answer:")
    print(result["final_answer"])

    print()

CASE 1

Question:
How to change harmful behavior?

Route:
psychoeducation

Plan:
['search_knowledge_base', 'build_final_answer']

Completed steps:
['classify_request', 'build_plan', 'search_knowledge_base', 'build_final_answer']

Tool calls:
[{'tool': 'search_knowledge_base', 'input': {'query': 'How to change harmful behavior?'}}]

Retrieved chunk IDs:
- michie_2011_behaviour_change_wheel__0021
- ogden_2019_health_psychology__0068
- michie_2011_behaviour_change_wheel__0033

Final answer:
LLM synthesis is temporarily unavailable because the API quota was exceeded. Relevant evidence was successfully retrieved from the knowledge base.

CASE 2

Question:
Why can't I work effectively with chronic pain?

Route:
psychoeducation

Plan:
['search_knowledge_base', 'build_final_answer']

Completed steps:
['classify_request', 'build_plan', 'search_knowledge_base', 'build_final_answer']

Tool calls:
[{'tool': 'search_knowledge_base', 'input': {'query': "Why can't I work effectively with chronic pain

**Stage 8 result:** The workflow was successfully tested across five scenarios. Routing, retrieval, medical boundaries, IVR referral, clarification, and state tracking work as expected. LLM synthesis is used only to make retrieved evidence readable. A fallback was added so the workflow remains stable when the Gemini API quota is exceeded.